In [1]:
import argparse
from neural_priors.utils.data import Subject
from fit_model import get_paradigm, get_model, fit_model, fit_model_cv
from pathlib import Path
import numpy as np
from braincoder.utils import get_rsq
import pandas as pd
import os
import os.path as op
from braincoder.optimize import ParameterFitter
from braincoder.models import AlphaGaussianPRF
from neural_priors.encoding_model2.models import AlphaDeltaModel
from braincoder.optimize import ResidualFitter
import pingouin as pg

In [13]:
def get_decoding_paradigm(sub, fit_responses=False, drop_levels=True):
        # Get paradigm/data/model
    paradigm = get_paradigm(sub, fit_responses=fit_responses)
    paradigm = paradigm.set_index(pd.Index((paradigm.index.get_level_values('run') - 1) % 4 + 1, name='run2'), append=True)
    paradigm.index = paradigm.index.swaplevel('run', 'run2')
    paradigm = paradigm.astype(np.float32)
    
    if drop_levels:
        paradigm = paradigm.droplevel(['run', 'trial_nr', 'subject'])
    
    return paradigm

# def main(subject, model_label=3, roi='NPCr', bids_folder='/data/ds-neural_priors', smoothed=True, debug=False, fit_responses=False,
#          n_voxels=100):

bids_folder = '/data/ds-neuralpriors'

subject = 1
model_label = 15
debug= True
smoothed = True
fit_responses = False

separate_sigmas = True

roi = 'NPCr'
n_voxels = 100

assert model_label in [15], 'Only model 3, 4 and 5 are supported for decoding'


sub = Subject(subject_id=subject, bids_folder=bids_folder)
bids_folder = Path(bids_folder)

max_n_iterations = 100 if debug else 2000

# Create target folder
key = f'model{model_label}'

if smoothed:
    key += '.smoothed'

if fit_responses:
    key += '.fit_responses'

if separate_sigmas:
    key += '.seperate_sigmas'

target_dir = bids_folder / 'derivatives' / 'decoding2' / key / f'sub-{subject}' / 'func'

if not op.exists(target_dir):
    os.makedirs(target_dir)

# Get paradigm/data/model
paradigm = get_decoding_paradigm(sub, fit_responses=fit_responses)

data = sub.get_single_trial_estimates(session=None, smoothed=smoothed)
masker = sub.get_volume_mask(roi=roi, epi_space=True, return_masker=True)
data = pd.DataFrame(masker.fit_transform(data), index=paradigm.index).astype(np.float32)

# all_cvr2 = []

stimulus_range = np.sort(paradigm['x'].unique())
stimulus_range = np.stack([np.repeat(stimulus_range, 2), np.stack(np.tile([0, 1], len(stimulus_range)), axis=0)], axis=1)

pdfs = []

for (test_session, test_run), _ in paradigm.groupby(level=['session', 'run2']):

    print(f'Fitting using session {test_session} run {test_run} as test set')

    test_data, test_paradigm = data.loc[(test_session, test_run)].copy().astype(np.float32), paradigm.loc[(test_session, test_run)].copy().astype(np.float32)
    train_data, train_paradigm = data.drop((test_session, test_run)).copy(), paradigm.drop((test_session, test_run)).copy()

    print(test_paradigm)

    # Get model
    model = get_model(model_label)

    # Cross-validate to get number of (and which) voxels
    if n_voxels == 0:

        print('Cross-validating to get number of voxels')

        cvr2 = fit_model_cv(train_data, train_paradigm, model_label, max_n_iterations=max_n_iterations)
        print(cvr2)
        r2_mask = cvr2 > 0.0

        target_fn = op.join(target_dir, f'sub-{subject}_ses-{test_session}_run2-{test_run}_mask-{roi}_desc-cvr2_pars.tsv')
        cvr2.to_csv(target_fn, sep='\t')

        print(f'Selecting {np.sum(r2_mask)} voxels with cvr2 > 0.0')

        print(train_data)
        train_data = train_data.loc[:, r2_mask]
        print(train_data)
        test_data = test_data.loc[:, r2_mask]
        gd_pars = fit_model(model_label, model, train_data.loc[:, r2_mask], train_paradigm, max_n_iterations=max_n_iterations)        

    else:

        gd_pars = fit_model(model_label, model, train_data, train_paradigm, max_n_iterations=max_n_iterations)        

        pred = model.predict(paradigm=train_paradigm, parameters=gd_pars)

        r2 = get_rsq(train_data, pred)
        print(r2.describe())

        r2 = r2[r2 < 1.0]
        r2_mask = r2.sort_values(ascending=False).index[:n_voxels]

        gd_pars = gd_pars.loc[r2_mask]
        model.apply_mask(r2_mask)

        train_data = train_data[r2_mask]
        test_data = test_data[r2_mask]


    if separate_sigmas:

        train_narrow_ix = train_paradigm['range'] == 0.0
        train_wide_ix = train_paradigm['range'] == 1.0

        train_data_narrow = train_data[train_narrow_ix]
        train_data_wide = train_data[train_wide_ix]
        train_paradigm_narrow = train_paradigm[train_narrow_ix]
        train_paradigm_wide = train_paradigm[train_wide_ix]

        
        test_narrow_ix = test_paradigm['range'] == 0.0
        test_wide_ix = test_paradigm['range'] == 1.0
        test_data_narrow = test_data[test_narrow_ix]
        test_data_wide = test_data[test_wide_ix]


        stimulus_range_narow = stimulus_range[:len(stimulus_range) // 2]
        stimulus_range_wide = stimulus_range[len(stimulus_range) // 2:]

        # Get pdf for narrow condition
        model.init_pseudoWWT(stimulus_range_narow, gd_pars)

        residfit_narrow = ResidualFitter(model, train_data_narrow,
                                        train_paradigm_narrow, parameters=gd_pars,)
        omega_narrow, dof_narrow = residfit_narrow.fit(init_sigma2=0.1,
                init_dof=10.0,
                method='t',
                learning_rate=0.05,
                max_n_iterations=20000 if not debug else 100)

        print('DOF narrow', dof_narrow)
        pdf_narrow = model.get_stimulus_pdf(test_data_narrow, stimulus_range,
                gd_pars,
                omega=omega_narrow,
                dof=dof_narrow,
                normalize=False)
        

        # Get pdf for wide condition
        model.init_pseudoWWT(stimulus_range_wide, gd_pars)
        residfit_wide = ResidualFitter(model, train_data_wide,
                                        train_paradigm_wide, parameters=gd_pars,)
        omega_wide, dof_wide = residfit_wide.fit(init_sigma2=0.1,
                init_dof=10.0,
                method='t',
                learning_rate=0.05,
                max_n_iterations=20000 if not debug else 100)

        print('DOF wide', dof_wide)
        pdf_wide = model.get_stimulus_pdf(test_data_wide, stimulus_range,
                gd_pars,
                omega=omega_wide,
                dof=dof_wide,
                normalize=False)
        pdfs.append(pd.concat([pdf_narrow, pdf_wide], axis=1))



    else:
        model.init_pseudoWWT(stimulus_range, gd_pars)

        residfit = ResidualFitter(model, train_data,
                                    train_paradigm, parameters=gd_pars,)

        omega, dof = residfit.fit(init_sigma2=0.1,
                init_dof=10.0,
                method='t',
                learning_rate=0.05,
                max_n_iterations=20000 if not debug else 100)

        print('DOF', dof)

        pdf = model.get_stimulus_pdf(test_data, stimulus_range,
                gd_pars,
                omega=omega,
                dof=dof,
                normalize=False)

        print(pdf)
        pdfs.append(pdf)

pdfs = pd.concat(pdfs)        

target_fn = op.join(target_dir, f'sub-{subject}_mask-{roi}_nvoxels-{n_voxels}_pars.tsv')

Fitting using session 1 run 1 as test set
                 x  range
session run2             
1       1     11.0    0.0
        1     23.0    0.0
        1     23.0    0.0
        1     22.0    0.0
        1     21.0    0.0
        1     20.0    0.0
        1     10.0    0.0
        1     19.0    0.0
        1     21.0    0.0
        1     18.0    0.0
        1     22.0    0.0
        1     13.0    0.0
        1     20.0    0.0
        1     19.0    0.0
        1     19.0    0.0
        1     20.0    0.0
        1     20.0    0.0
        1     12.0    0.0
        1     12.0    0.0
        1     16.0    0.0
        1     23.0    0.0
        1     10.0    0.0
        1     18.0    0.0
        1     16.0    0.0
        1     11.0    0.0
        1     12.0    0.0
        1     20.0    0.0
        1     17.0    0.0
        1     10.0    0.0
        1     22.0    0.0
        1     28.0    1.0
        1     34.0    1.0
        1     19.0    1.0
        1     33.0    1.0
        1     28.0    

/var/folders/d4/cpqhwlbn301clph1qk64k01nmjkty3/T/ipykernel_52495/151567345.py:72: PerformanceWarning: indexing past lexsort depth may impact performance.
  test_data, test_paradigm = data.loc[(test_session, test_run)].copy().astype(np.float32), paradigm.loc[(test_session, test_run)].copy().astype(np.float32)
/var/folders/d4/cpqhwlbn301clph1qk64k01nmjkty3/T/ipykernel_52495/151567345.py:72: PerformanceWarning: indexing past lexsort depth may impact performance.
  test_data, test_paradigm = data.loc[(test_session, test_run)].copy().astype(np.float32), paradigm.loc[(test_session, test_run)].copy().astype(np.float32)


  0%|          | 0/1 [00:00<?, ?it/s]

*** Fitting: ***
 * mu_narrow
 * baseline
 * sd_narrow
 * sd_wide_scale
 * amplitude
*** Fixed Parameters: ***
 * lower_bound_range
 * delta_wide
*** Shared Parameters: ***
 * sd_wide_scale
Number of problematic voxels (mask): 0
Number of voxels remaining (mask): 771


Current R2: -0.01686/Best R2: -0.01686: 100%|██████████| 100/100 [00:01<00:00, 62.30it/s]


...

count    771.000000
mean      -0.021185
std        0.048520
min       -0.397141
25%       -0.026627
50%       -0.000823
75%        0.002839
max        0.047612
Name: r2, dtype: float64
init_tau: 0.2872438132762909, 1.0112258195877075
USING A PSEUDO-WWT!
WWT max: 36.044063568115234


  0%|          | 0/100 [00:00<?, ?it/s]

DOF narrow 9.938117
init_tau: 0.3280123770236969, 1.201945424079895
USING A PSEUDO-WWT!
WWT max: 31.862911224365234


  0%|          | 0/100 [00:00<?, ?it/s]

DOF wide 8.662491


In [12]:
test_data

305       303       306       367       370       250  \
session run2                                                               
1       1     1.287796  1.625972  0.984352  1.427470  1.153153  0.874286   
        1     0.143345 -0.170333  0.319031 -0.029559  0.377120  0.298203   
        1     0.074736 -0.287898  0.417389 -0.232064  0.509764 -0.150329   
        1     0.595963  1.121234  0.626036  0.863480  0.961999  0.547039   
        1    -0.291656 -0.437774 -0.068384 -0.536488 -0.031253 -0.085293   
        1     1.175819  1.112644  0.763139  0.977652  0.704096  0.810695   
        1     1.221768  0.994029  1.268314  0.284468  1.105377  1.185056   
        1     0.555407  0.955407  0.708430  0.432017  0.703367  0.653574   
        1     0.306929  0.109070  0.204743  0.194848  0.272603  0.046067   
        1     0.984464  1.476715  0.686375  1.004672  0.625522  1.168666   
        1     0.172300  0.340128  0.352709  0.144116  0.577287  0.242986   
        1     0.881557  0.959119  1.104026  0.628215  1.146143  0.835685   
        1     0.449823  0.932847  0.199518  0.840682  0.357824  0.297236   
        1     0.738084  0.959410  0.409329  0.546015  0.312918  0.529720   
        1     0.534251  0.746422  0.568009  0.621244  0.369152  0.412627   
        1     0.384643  0.332197  0.682318  0.454676  0.657758  0.511467   
        1     0.257693  0.252783  0.272186  0.207689  0.362468 -0.033134   
        1     0.797720  1.008544  0.682813  0.886306  0.509117  0.853747   
        1     1.051883  1.450676  0.846555  1.105214  0.816989  0.774844   
        1    -0.335387 -0.007068 -0.128888 -0.126304  0.036381 -0.051034   
        1    -0.494056 -0.183519 -0.183319 -0.405236 -0.199248 -0.134228   
        1     1.472028  1.973662  0.945949  1.268547  1.048090  1.188596   
        1     1.010452  1.046654  1.005807  0.804984  1.156545  0.731355   
        1     1.320459  1.356954  0.433607  1.244782  0.400671  0.927516   
        1     0.864970  1.141088  1.087802  0.816602  1.132636  1.255866   
        1     1.355256  1.675350  0.780994  0.849053  0.756254  0.986070   
        1     0.286469  0.103492  0.793660 -0.214517  0.733670  0.232677   
        1     0.371193  0.821044  0.472210  0.325298  0.527548  0.406261   
        1     0.149280 -0.113466  0.429181 -0.151856  0.607100  0.010801   
        1     0.705833  0.793760  0.342233  0.611413  0.368185  0.108908   
        1     1.341627  1.334216  1.158091  1.017394  1.238012  1.116425   
        1     1.800111  2.506956  1.116650  1.794717  1.200947  1.477687   
        1     1.405220  1.648010  0.885599  1.163713  0.945010  0.876489   
        1     1.232476  1.541272  1.320038  1.176172  1.254829  1.463043   
        1     0.608760  0.725447  0.714411  0.660101  0.666265  0.533197   
        1     0.289772  0.587097  0.529025  0.342043  0.349850  0.587051   
        1     1.171387  1.227339  0.859101  0.891187  0.907629  0.842900   
        1    -0.113029  0.119395 -0.131485 -0.007186  0.053215 -0.107346   
        1     0.568769  0.866851  0.605352  0.430043  0.577604  0.764169   
        1     0.357770  0.133970  0.559820  0.133133  0.636198  0.142288   
        1     1.015305  0.977302  0.946087  0.612929  1.018590  0.889698   
        1     0.212707  0.312247  0.443731  0.243779  0.750811  0.226281   
        1     1.641268  1.959699  1.094088  1.409263  1.315691  1.296440   
        1     0.577431  0.807599  0.804376  0.617737  0.468761  0.666257   
        1     0.481964  0.606543  0.833054  0.201353  0.739713  0.962316   
        1    -0.369935  0.090738 -0.578028  0.299648 -0.566498 -0.050594   
        1     0.292926  0.530276  0.183956  0.456805  0.245608 -0.101664   
        1     0.667792  1.290681  0.191125  0.965677  0.285354  0.150045   
        1     0.976252  1.022984  0.829812  0.665735  0.786845  0.897066   
        1    -0.113303 -0.129904  0.167571  0.140346  0.349765 -0.188613   
        1    -0.000078 -0.040304  0.180648 -0.220292  0.317654  0.0

NameError: name 'pdf' is not defined

In [8]:
train_paradigm

x  range
session run2             
1       2     11.0    0.0
        2     18.0    0.0
        2     24.0    0.0
        2     17.0    0.0
        2     17.0    0.0
...            ...    ...
2       4     12.0    0.0
        4     13.0    0.0
        4     18.0    0.0
        4     14.0    0.0
        4     17.0    0.0

[420 rows x 2 columns]

In [7]:
train_paradigm

x  range
session run2             
1       2     11.0    0.0
        2     18.0    0.0
        2     24.0    0.0
        2     17.0    0.0
        2     17.0    0.0
...            ...    ...
2       4     12.0    0.0
        4     13.0    0.0
        4     18.0    0.0
        4     14.0    0.0
        4     17.0    0.0

[420 rows x 2 columns]

In [14]:
pdfs

dim_0             10.0                11.0                12.0            \
dim_1              0.0       1.0       0.0       1.0       0.0       1.0   
session run2                                                               
1       1     0.000053  0.000043  0.000027  0.000027  0.000024  0.000031   
        1     0.004676  0.009441  0.026461  0.059301  0.074684  0.190549   
        1     0.004421  0.003171  0.021441  0.015327  0.063932  0.046268   
        1     0.246172  0.385067  0.735707  1.000000  0.790908  0.877924   
        1     0.871138  0.483812  1.000000  0.516471  0.821254  0.386863   
...                ...       ...       ...       ...       ...       ...   
2       4     0.090663  0.046281  0.206838  0.168466  0.332732  0.394508   
        4     1.000000  0.091802  0.573700  0.076577  0.292182  0.050244   
        4     0.001600  0.005150  0.000283  0.000602  0.000100  0.000128   
        4     0.024651  0.055710  0.039440  0.082351  0.055373  0.091391   
        4     0.003895  0.018007  0.030255  0.154539  0.110313  0.492384   

dim_0             13.0                14.0            ...          36.0  \
dim_1              0.0       1.0       0.0       1.0  ...           0.0   
session run2                                          ...                 
1       1     0.000033  0.000059  0.000057  0.000148  ...  1.079455e-03   
        1     0.135872  0.408590  0.180236  0.664895  ...  4.197298e-07   
        1     0.146558  0.107710  0.281081  0.208831  ...  2.337745e-06   
        1     0.493599  0.431255  0.236407  0.159324  ...  1.816075e-04   
        1     0.590220  0.246711  0.409236  0.147076  ...  4.177695e-08   
...                ...       ...       ...       ...  ...           ...   
2       4     0.426787  0.670007  0.468062  0.896203  ...  4.483311e-01   
        4     0.170598  0.035706  0.120531  0.030447  ...  5.370385e-02   
        4     0.000066  0.000051  0.000070  0.000034  ...  2.591616e-01   
        4     0.075791  0.092224  0.104676  0.093242  ...  1.035864e-01   
        4     0.247035  0.843705  0.401927  1.000000  ...  8.522950e-05   

dim_0                               37.0                        38.0  \
dim_1                  1.0           0.0           1.0           0.0   
session run2                                                           
1       1     3.781595e-02  9.007633e-04  3.432375e-02  7.600480e-04   
        1     9.366835e-07  2.456907e-07  5.571727e-07  1.476045e-07   
        1     5.622248e-05  2.007681e-06  4.805223e-05  1.760131e-06   
        1     1.966091e-02  1.653681e-04  1.910667e-02  1.525048e-04   
        1     2.389388e-06  3.394370e-08  1.965787e-06  2.827331e-08   
...                    ...           ...           ...           ...   
2       4     7.219455e-01  4.775228e-01  7.909806e-01  5.067096e-01   
        4     9.324012e-01  4.814900e-02  8.794588e-01  4.298828e-02   
        4     9.928768e-01  2.520153e-01  1.000000e+00  2.425235e-01   
        4     4.434239e-02  8.520330e-02  3.645252e-02  7.019870e-02   
        4     7.377462e-05  7.466134e-05  6.384563e-05  6.578392e-05   

dim_0                               39.0                        40.0  \
dim_1                  1.0           0.0           1.0           0.0   
session run2                                                           
1       1     3.063597e-02  6.488846e-04  2.693527e-02  5.607065e-04   
        1     3.396690e-07  9.091431e-08  2.120670e-07  5.734419e-08   
        1     4.101041e-05  1.574405e-06  3.496477e-05  1.435767e-06   
        1     1.806969e-02  1.426085e-04  1.667022e-02  1.353118e-04   
        1     1.615064e-06  2.411748e-08  1.325438e-06  2.104527e-08   
...                    ...           ...           ...           ...   
2       4     8.605491e-01  5.359806e-01  9.303192e-01  5.653746e-01   
        4     8.184959e-01  3.826725e-02  7.532058e-01  3.399849e-02   
        4     9.881913e-01  2.314140e-01  9.607739e-01  2.192974e-01   
        4  